# Experiment 4 — Peripheral EOG Controls

## Scientific question

Can the observed facial-video → EEG correspondence be explained by
peripheral ocular activity?

The analysis has two complementary components:

1. Signal-level EEG–EOG coupling:
   quantify the association between the EEG target and peripheral
   hEOG/vEOG signals.

2. EOG→EEG linear-regression baseline:
   predict the target EEG channel from hEOG and vEOG alone under
   the same five-fold within-participant GroupKFold structure used
   by the facial-video model.

The primary comparison is therefore:

`Facial video → EEG` versus `hEOG + vEOG → EEG`

## Dataset

- 18 complete participants
- 40 trials per participant
- 60 s per trial
- EEG sampling rate: 128 Hz
- Video sampling rate: 50 Hz

## EEG targets

- Fz
- Pz
- Oz

## EOG predictors

- hEOG
- vEOG

## Cross-validation

- 5-fold GroupKFold
- 8 held-out trials per fold
- same trial partition as the video models

## Primary metric

Fisher-z aggregated Pearson correlation.

## Secondary metrics

- RMSE
- theta coherence
- alpha coherence
- beta coherence
- RBP error

## Statistical hierarchy

The 40 held-out trials are the repeated-measures units within
each participant. The 18 participants are the independent units
for population-level inference.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display, HTML

sys.path.insert(0, r"D:\Thesis\Codes\Source")

from face2eeg.core.results import load_results_by_subject
from face2eeg.core.stats import (
    summarize_trials,
    compare_methods_trial_level,
    fisher_z,
    inv_fisher_z,
    sign_flip_pvalue,
    safe_pearsonr,
    population_cohens_dz,
    comparison_to_dataframes,
    bootstrap_ci_trial
)
from face2eeg.core.coupling import analyze_eeg_eog_emg_coupling
from face2eeg.core.plotting import (
    plot_coupling_heatmap_grid,
    plot_supplementary_coupling_alpha_beta,
    plot_population_level_coupling,
    plot_main_figure_video_vs_eog,
    plot_supplementary_population_level_coupling,
    plot_supplementary_spectral_heatmaps
)
from face2eeg.core.config import VALID_SUBJECTS, SEED, FS_EEG, FS_VIDEO

%matplotlib inline

## 1. Configuration

In [ ]:
BASE = Path(
    r"D:\Thesis\Codes\Data\Experiments\Cross-Validation"
)

VIDEO_DIRS = {
    "Fz": BASE / "Between-Participant-Variability" / "Results",
    "Pz": BASE / "Channel-Specificity" / "Results",
    "Oz": BASE / "Channel-Specificity" / "Results",
}

EOG_DIR = BASE / "EOG-Regression" / "Results"

EXP_DIR = BASE / "EOG-Regression"
FIG_DIR = EXP_DIR / "Results" / "Analysis-Figures"
TABLE_DIR = EXP_DIR / "Results" / "Analysis-Tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = list(VALID_SUBJECTS)
CHANNELS = ["Fz", "Pz", "Oz"]

N_TRIALS = 40
N_FOLDS = 5
N_TEST_PER_FOLD = 8

EEG_FS = FS_EEG
VIDEO_FS = FS_VIDEO
TRIAL_SECONDS = 60
TRIAL_SAMPLES = EEG_FS * TRIAL_SECONDS

N_BOOT = 10_000
N_PERM = 10_000

COUPLING_REFERENCES = ("heog", "veog")
COUPLING_BANDS = ("theta", "alpha", "beta")

print(f"N subjects: {len(SUBJECTS)}")
print(f"Subjects: {SUBJECTS}")
print("Channels :", CHANNELS)
print("Trials   :", N_TRIALS)
print("Folds    :", N_FOLDS)
print("Trial samples :", TRIAL_SAMPLES)
print("EOG results :", EOG_DIR)

## 2. Input-file audit

Every subject must have:

- one facial-video result for each target channel
- one EOG-regression result for each target channel

In [ ]:
for ch in CHANNELS:
    missing_video = [
        str(VIDEO_DIRS[ch] / f"sub{sid:02d}_ch{ch}.pkl")
        for sid in SUBJECTS
        if not (VIDEO_DIRS[ch] / f"sub{sid:02d}_ch{ch}.pkl").exists()
    ]
    if missing_video:
        raise FileNotFoundError(
            f"{ch}: missing video-model files:\n" +
            "\n".join(missing_video)
        )

missing_eog = [
    str(EOG_DIR / f"sub{sid:02d}_ch{ch}.pkl")
    for sid in SUBJECTS
    for ch in CHANNELS
    if not (EOG_DIR / f"sub{sid:02d}_ch{ch}.pkl").exists()
]

if missing_eog:
    raise FileNotFoundError(
        "Missing EOG-regression files:\n" + "\n".join(missing_eog)
    )

print("All required files are present.")

## 3. Load prediction results

Fz facial-video results are read from the final
Between-Participant-Variability experiment.

Pz/Oz facial-video results are read from Channel-Specificity.

All EOG→EEG results are read from EOG-Regression.

In [ ]:
video_results = {}
eog_results = {}

for ch in CHANNELS:
    video_results[ch] = load_results_by_subject(
        str(VIDEO_DIRS[ch]),
        SUBJECTS,
        channel_name=ch,
        pred_key="y_pred",
    )

    eog_results[ch] = load_results_by_subject(
        str(EOG_DIR),
        SUBJECTS,
        channel_name=ch,
        pred_key="y_pred",
    )

    assert set(video_results[ch]) == set(SUBJECTS)
    assert set(eog_results[ch]) == set(SUBJECTS)

print("Video model and EOG regression results loaded.")

## 4. Metadata and fold-alignment audit

The EOG regression must use the same participant, channel, trial
partition, and GroupKFold structure as the corresponding video model.

The EOG regression itself uses whole 60-s trials (`time_window=60`,
segmentation=False), whereas the video model reconstructs complete
60-s trials from its 2-s test segments.

In [ ]:
audit_rows = []

for ch in CHANNELS:
    for sid in SUBJECTS:

        rv = video_results[ch][sid]
        re = eog_results[ch][sid]

        assert rv.channel == ch
        assert re.channel == ch

        assert rv.n_splits == N_FOLDS
        assert re.n_splits == N_FOLDS

        assert float(rv.time_window) == 2.0
        assert float(re.time_window) == 60.0

        assert re.config.get("segmentation") is False
        assert re.config.get("references") == ["heog", "veog"]
        assert re.config.get("zscore") == "per-fold, train statistics only"

        assert rv.fold_ids() == re.fold_ids()

        for fold in rv.fold_ids():
            tv = np.asarray(rv.trials(fold))
            te = np.asarray(re.trials(fold))

            if not np.array_equal(tv, te):
                raise ValueError(
                    f"S{sid:02d} {ch} fold {fold}: "
                    "video model and EOG regression trial partitions differ."
                )

            assert len(tv) == N_TEST_PER_FOLD

        audit_rows.append({
            "Subject": sid,
            "Channel": ch,
            "Video model folds": rv.n_splits,
            "EOG folds": re.n_splits,
            "Video model T (s)": rv.time_window,
            "EOG trial length (s)": re.time_window,
            "EOG segmentation": re.config.get("segmentation"),
            "EOG z-score": re.config.get("zscore"),
        })

df_audit = pd.DataFrame(audit_rows)
display(df_audit.head())
print("Fold/trial alignment verified for all subjects and channels.")

## 5. Complete held-out trial arrays

The video model produces segmented predictions, which are reconstructed
into complete 60-s trials.

The EOG regression already produces complete 60-s predictions.

Therefore both analyses are evaluated on:

    (40 trials, 7680 samples)

In [ ]:
trial_data = {}

for ch in CHANNELS:
    trial_data[ch] = {}

    for sid in SUBJECTS:

        rv = video_results[ch][sid]
        re = eog_results[ch][sid]

        yv, pv = rv.get_all_trials()
        ye, pe = re.get_all_trials()

        ids_v = rv.get_trial_indices()
        ids_e = re.get_trial_indices()

        if not np.array_equal(ids_v, ids_e):
            raise ValueError(
                f"S{sid:02d} {ch}: trial IDs are not aligned."
            )

        if yv.shape != (N_TRIALS, TRIAL_SAMPLES):
            raise ValueError(
                f"S{sid:02d} {ch}: unexpected video target shape {yv.shape}"
            )

        if ye.shape != (N_TRIALS, TRIAL_SAMPLES):
            raise ValueError(
                f"S{sid:02d} {ch}: unexpected EOG target shape {ye.shape}"
            )

        if pv.shape != yv.shape or pe.shape != ye.shape:
            raise ValueError(
                f"S{sid:02d} {ch}: prediction/target shape mismatch."
            )

        trial_data[ch][sid] = {
            "trial_ids": ids_v,
            "video": {"y_true": yv, "y_pred": pv},
            "eog": {"y_true": ye, "y_pred": pe},
        }

print("Complete trial arrays reconstructed.")
# ------------------------------------------------------------------
# Cross-channel trial-ID identity for the same subject
# ------------------------------------------------------------------
for sid in SUBJECTS:
    ids_by_ch = {
        ch: np.asarray(trial_data[ch][sid]["trial_ids"])
        for ch in CHANNELS
    }
    if not (
        np.array_equal(ids_by_ch["Fz"], ids_by_ch["Pz"])
        and np.array_equal(ids_by_ch["Fz"], ids_by_ch["Oz"])
    ):
        raise ValueError(
            f"S{sid:02d}: trial IDs differ across channels "
            f"(Fz vs Pz vs Oz). Cross-channel comparison would "
            f"pair mismatched trials."
        )

print("Cross-channel trial-ID identity verified for all subjects.")

## 6. Prediction QC (internal validation only)

This table verifies that predictions are finite, have reasonable
scale, and no silent NaNs. It is an internal audit and is NOT
reported in the manuscript or supplementary material.

In [ ]:
qc_rows = []

for ch in CHANNELS:
    for sid in SUBJECTS:

        for predictor in ("Video model", "EOG regression"):

            d = trial_data[ch][sid][
                "video" if predictor == "Video model" else "eog"
            ]

            y = d["y_true"]
            p = d["y_pred"]

            qc_rows.append({
                "Subject": sid,
                "Channel": ch,
                "Predictor": predictor,
                "N trials": y.shape[0],
                "Samples/trial": y.shape[1],
                "Target finite ratio": np.isfinite(y).mean(),
                "Prediction finite ratio": np.isfinite(p).mean(),
                "Prediction mean": np.nanmean(p),
                "Prediction SD": np.nanstd(p),
                "Prediction min": np.nanmin(p),
                "Prediction max": np.nanmax(p),
            })

df_prediction_qc = pd.DataFrame(qc_rows)
display(df_prediction_qc.round(4))

## 7. Signal-level EEG–EOG coupling

This analysis does not train a model.

For every participant and target channel, the existing
`analyze_eeg_eog_emg_coupling()` function is applied to the same
40 DEAP trials.

Primary diagnostic quantities:

- Pearson r between EEG and hEOG/vEOG
- theta-band coherence

Alpha and beta coherence are retained for supplementary analysis.

zEMG is treated as a secondary peripheral diagnostic.

## 7.1. Computation of EEG–EOG coupling
Signal-level EEG-EOG coupling

Two distinct quantities are computed:

1. Broadband Pearson r (4-45 Hz), band-independent
    - Stored in `coupling_broadband_r`
    - One value per (subject, channel, reference)
    - No frequency band is imposed on this quantity

2. Band-averaged coherence per band (theta, alpha, beta)
    - Stored in `coupling_results[band][ch][sid]["_coh"]`
    - Fisher-z is NOT applied to coherence
    - Each band is treated as an independent metric

In [ ]:
coupling_results = {
    band: {ch: {} for ch in CHANNELS}
    for band in COUPLING_BANDS
}

# NEW: broadband r stored separately, not nested in bands
coupling_broadband_r = {
    ch: {sid: {} for sid in SUBJECTS}
    for ch in CHANNELS
}

coupling_spectra = {ch: {} for ch in CHANNELS}
coupling_freqs = {}

for ch in CHANNELS:
    for sid in SUBJECTS:

        res = analyze_eeg_eog_emg_coupling(
            eeg_channel_name=ch,
            participant_id=sid,
            experiment_ids=np.arange(N_TRIALS),
            fs=EEG_FS,
            nperseg=1024,
            bands=tuple(COUPLING_BANDS),
            references=tuple(COUPLING_REFERENCES),
            root_dir=str(BASE.parent.parent),
        )

        # ---- broadband r (band-independent) ----
        for ref in COUPLING_REFERENCES:
            coupling_broadband_r[ch][sid][ref] = res[f"eeg_{ref}_r"]

        # ---- per-band coherence only ----
        for band in COUPLING_BANDS:
            coupling_results[band][ch][sid] = {
                "_coh": {
                    ref: res[f"eeg_{ref}_{band}_coh"]
                    for ref in COUPLING_REFERENCES
                },
            }

        coupling_spectra[ch][sid] = {
            ref: res[f"coh_{ref}_spectra"]
            for ref in COUPLING_REFERENCES
        }
        coupling_freqs[ch] = res["f_coh"]

print("EEG–EOG coupling analysis completed.")

## 7.2. Subject-level coupling estimates

In [ ]:
# ============================================================
# Wide-format coupling table
# ============================================================
# One row per (Subject, Target, Reference).
# Columns:
#   - r_broadband_4_45 : Fisher-z aggregated across 40 trials
#   - coh_theta        : mean across 40 trials (linear scale)
#   - coh_alpha        : mean across 40 trials (linear scale)
#   - coh_beta         : mean across 40 trials (linear scale)
# ============================================================

coupling_rows = []

for ch in CHANNELS:
    for sid in SUBJECTS:
        for ref in COUPLING_REFERENCES:

            # ---- initialize the row ----
            row = {
                "Subject":   f"S{sid:02d}",
                "Target":    ch,
                "Reference": ref,
            }

            # ---- broadband r (independent of band) ----
            r_raw = np.asarray(
                coupling_broadband_r[ch][sid][ref], dtype=float
            )
            r_raw = r_raw[np.isfinite(r_raw)]
            if r_raw.size:
                z = fisher_z(r_raw)
                row["r_broadband_4_45"] = float(inv_fisher_z(np.mean(z)))
            else:
                row["r_broadband_4_45"] = np.nan

            # ---- per-band coherence (Fisher-z NOT applied) ----
            for band in COUPLING_BANDS:
                coh = np.asarray(
                    coupling_results[band][ch][sid]["_coh"][ref],
                    dtype=float,
                )
                coh = coh[np.isfinite(coh)]
                row[f"coh_{band}"] = (
                    float(np.mean(coh)) if coh.size else np.nan
                )

            # ---- append to list ----
            coupling_rows.append(row)

# ---- build DataFrame ----
df_coupling_wide = (
    pd.DataFrame(coupling_rows)
    .sort_values(["Target", "Reference", "Subject"])
    .reset_index(drop=True)
)

df_coupling_wide.to_csv(
    TABLE_DIR / "Table_S9_df_coupling_wide.csv",
    index=False,
)
display(df_coupling_wide.round(4))

In [ ]:
fig = plot_coupling_heatmap_grid(
    df_coupling_wide,
    channels=("Fz", "Pz", "Oz"),
    references=("heog", "veog"),
    metrics=("r_broadband_4_45", "coh_theta"),
    metric_titles={
        "r_broadband_4_45": "Broadband Pearson r (4\u201345 Hz)",
        "coh_theta":        "Theta coherence (4\u20138 Hz)",
    },
    cmaps={
        "r_broadband_4_45": "RdBu_r",
        "coh_theta": "viridis",
    },
    value_ranges={
        "r_broadband_4_45": (-1.0, 1.0),
        "coh_theta":        (0.0, 1.0),
    },
    save_fig=True,
    file_name=str(FIG_DIR / "fig_coupling_heatmap_all_channels_and_subjects.pdf"),
)

plt.show()
plt.close(fig)

In [ ]:
fig = plot_supplementary_coupling_alpha_beta(
    df=df_coupling_wide,
    channels=("Fz", "Pz", "Oz"),
    references=("heog", "veog"),
    metrics=("coh_alpha", "coh_beta"),
    cmaps={
        "coh_alpha": "viridis",
        "coh_beta": "viridis",
    },
    save_fig=True,
    file_name=str(FIG_DIR / "fig_S_EEG_EOG_alpha_beta_coupling.pdf"),
)

plt.show()
plt.close(fig)

### 7.3. Population-level coupling aggregation (N = 18 participants)

 Input : df_coupling_wide  (one row per Subject × Target × Reference)

         columns: Subject, Target, Reference, r, coh_theta, coh_alpha, coh_beta

 For each (Target, Reference) pair:

   - r         : Fisher-z aggregated across 18 subjects, back-transformed

   - coh_theta : linear mean across 18 subjects (primary frequency metric)

   - coh_alpha : linear mean (secondary)

   - coh_beta  : linear mean (secondary)

 Statistics per pair and metric:
   mean, 95% percentile-bootstrap CI, sign-flip permutation p,

   Wilcoxon p, Cohen's dz, Holm-adjusted p.

In [ ]:
def _aggregate_coupling(values, scale="linear"):
    """
    Aggregate one metric across the 18 independent participants.

    scale = "fisher" : atanh → mean → tanh (for correlation)
    scale = "linear" : raw scale (for coherence)

     Notes
    -----
    - Fisher-z is applied ONLY to Pearson r, never to coherence.
    - Coherence is bounded in [0, 1] and is not a correlation;
      applying atanh to it would be statistically unjustified.
      
    """
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    n = v.size

    if n == 0:
        return dict(n=0, mean=np.nan, ci_lo=np.nan, ci_hi=np.nan,
                    p_perm=np.nan, p_wilcox=np.nan, dz=np.nan)

    # Transform if Fisher
    v_t = fisher_z(v) if scale == "fisher" else v
    mean_t = float(np.mean(v_t))

    if n > 1:
        boot = stats.bootstrap(
            (v_t,), np.mean,
            confidence_level=0.95,
            n_resamples=N_BOOT,
            method="percentile",
            random_state=SEED,
        )
        ci_lo_t = float(boot.confidence_interval.low)
        ci_hi_t = float(boot.confidence_interval.high)

        p_perm = float(sign_flip_pvalue(v_t, n_perm=N_PERM, seed=SEED))

        try:
            p_wilcox = float(stats.wilcoxon(
                v_t, alternative="two-sided", zero_method="wilcox"
            ).pvalue)
        except ValueError:
            p_wilcox = np.nan

        sd = float(np.std(v_t, ddof=1))
        dz = mean_t / sd if sd > 0 else np.nan
    else:
        ci_lo_t = ci_hi_t = mean_t
        p_perm = p_wilcox = dz = np.nan

    # Back-transform for display
    if scale == "fisher":
        mean_disp = float(inv_fisher_z(mean_t))
        ci_lo_disp = float(inv_fisher_z(ci_lo_t))
        ci_hi_disp = float(inv_fisher_z(ci_hi_t))
    else:
        mean_disp = mean_t
        ci_lo_disp = ci_lo_t
        ci_hi_disp = ci_hi_t

    return dict(
        n=n,
        mean=mean_disp,
        ci_lo=ci_lo_disp,
        ci_hi=ci_hi_disp,
        p_perm=p_perm,
        p_wilcox=p_wilcox,
        dz=dz,
    )


# ------------------------------------------------------------
# Metric definitions
# ------------------------------------------------------------
METRICS = [
    # (column_name,        display_label,          family,        scale)
    ("r_broadband_4_45", "r (4\u201345 Hz)",     "correlation", "fisher"),
    ("coh_theta",        r"$\theta$-coherence",  "coherence",   "linear"),
    ("coh_alpha",        r"$\alpha$-coherence",  "coherence",   "linear"),
    ("coh_beta",         r"$\beta$-coherence",   "coherence",   "linear"),
]


# ------------------------------------------------------------
# Aggregate across 18 participants per (Target, Reference, Metric)
# ------------------------------------------------------------
pop_rows = []

for ch in CHANNELS:
    for ref in COUPLING_REFERENCES:
        d = df_coupling_wide[
            (df_coupling_wide["Target"] == ch) &
            (df_coupling_wide["Reference"] == ref)
        ]

        for col, label, family, scale in METRICS:
            s = _aggregate_coupling(d[col].to_numpy(float), scale=scale)

            pop_rows.append({
                "Target":        ch,
                "Reference":     ref,
                "Metric":        label,
                "Family":        family,
                "N subjects":    s["n"],
                "Mean":          s["mean"],
                "CI_low":        s["ci_lo"],
                "CI_high":       s["ci_hi"],
                "Cohen_dz":      s["dz"],
                "Permutation_p": s["p_perm"],
                "Wilcoxon_p":    s["p_wilcox"],
            })

df_coupling_population = pd.DataFrame(pop_rows)


# ------------------------------------------------------------
# Holm correction within each metric family
# ------------------------------------------------------------
df_coupling_population["Holm_p"] = np.nan

for metric in df_coupling_population["Metric"].unique():
    mask = df_coupling_population["Metric"].eq(metric)
    p = df_coupling_population.loc[mask, "Permutation_p"].to_numpy(float)
    valid = np.isfinite(p)
    adj = np.full(len(p), np.nan)
    if valid.any():
        adj[valid] = multipletests(p[valid], method="holm")[1]
    df_coupling_population.loc[mask, "Holm_p"] = adj


# ------------------------------------------------------------
# Final ordering
# ------------------------------------------------------------
df_coupling_population = (
    df_coupling_population[[
        "Target", "Reference", "Metric", "Family",
        "N subjects", "Mean", "CI_low", "CI_high",
        "Cohen_dz", "Permutation_p", "Wilcoxon_p", "Holm_p",
    ]]
    .sort_values(["Family", "Target", "Reference"])
    .reset_index(drop=True)
)

display(df_coupling_population.round(5))
df_coupling_population.to_csv(
    TABLE_DIR / "Table_S2_coupling_population.csv",
    index=False,
)

In [ ]:
fig, axes = plot_population_level_coupling(
    df_wide=df_coupling_wide,
    df_population=df_coupling_population,
    references=COUPLING_REFERENCES,
    channels=CHANNELS,
    save_fig=True,
    file_name=str(FIG_DIR / "fig_coupling_main.pdf"),
)
plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# Supplementary Figure — coherence across all three bands
# ============================================================
fig, axes = plot_supplementary_population_level_coupling(
    df_wide       = df_coupling_wide,
    df_population = df_coupling_population,
    references    = COUPLING_REFERENCES,   # ("heog", "veog")
    channels      = CHANNELS,              # ("Fz", "Pz", "Oz")
    bands         = ("alpha", "beta"),
    save_fig      = True,
    file_name     = str(FIG_DIR / "fig_S_coupling_bands.pdf"),
)
plt.show()
plt.close(fig)

## 8. Paired Video vs EOG: Subject-Level Analysis

Purpose:

For every participant (N = 18) and every EEG target channel
(Fz, Pz, Oz), we compare two prediction pipelines under
identical held-out trials:

    A = Video model          (facial video → EEG)
    B = EOG regression       (hEOG + vEOG → EEG)

Metrics computed at the trial level (40 held-out trials per
subject and channel):

    - Fisher-z aggregated Pearson r   (waveform similarity)
    - Theta / alpha / beta coherence
    - Relative band-power (RBP) error

Statistical hierarchy:

Trials within a subject are treated as repeated measures
(paired across the two pipelines). The 18 participants are the
independent units of analysis. This produces subject-level
descriptive tables and paired-effect tables that feed into the
population-level aggregation in the next section.

Sign convention (all contrasts):

A − B = Video − EOG. Positive values indicate that the video
model outperforms the EOG regression on the corresponding metric.

### 8.1. Trial-level metric summaries


 For each (subject, target channel, predictor) combination, we
 compute trial-level metrics on the 40 held-out trials using the
 unmodified `summarize_trials()` function.

 This guarantees that both pipelines are evaluated with identical
 trial alignment and fold structure. All correlation, RMSE,
 coherence and RBP values used downstream in this notebook are
 extracted from the resulting structure.

Output:

`summaries[ch][sid][predictor]` :
    Nested dict produced by `summarize_trials` for each subject,
    channel, and predictor. Downstream cells extract:

- `r_pooled` / `r_pooled_ci_boot`    → Fisher-z aggregated
                                        Pearson r and its
                                        bootstrap CI
- `band_summary[band].coh`           → coherence
- `band_summary[band]._raw`          → per-trial arrays for
                                        RBP error

In [ ]:
summaries = {}

for ch in CHANNELS:
    summaries[ch] = {}
    for sid in SUBJECTS:
        summaries[ch][sid] = {
            "Video model": summarize_trials(
                trial_data[ch][sid]["video"]["y_true"],
                trial_data[ch][sid]["video"]["y_pred"],
                seed=SEED, 
                n_boot=N_BOOT
            ),
            "EOG regression": summarize_trials(
                trial_data[ch][sid]["eog"]["y_true"],
                trial_data[ch][sid]["eog"]["y_pred"],
                seed=SEED,
                n_boot=N_BOOT
            ),
        }

print(
    f"Trial-level summaries computed for "
    f"{len(CHANNELS)} channels × {len(SUBJECTS)} subjects × "
    f"2 predictors."
)

### 8.2. Trial-level paired comparison

For each (subject, target channel), we compare the video model
against the EOG regression on the **same 40 held-out trials**,
paired by trial index.

The paired contrast is computed in Fisher-z space for Pearson r:

    Δz(r) = z(r_Video) − z(r_EOG)

The comparison is performed at the **trial level**, because
trials within a subject are the repeated-measures units.

Output:

`paired_effects[ch][sid]` :
    Nested dict produced by `compare_methods_trial_level`,
    containing:

- `r`         → Δz(r) with its CI and permutation p
- `coherence` → per-band coherence
- `rbp_error` → per-band RBP error

The tidy DataFrames generated from this structure (via
`comparison_to_dataframes`) are used in Cells 8.6 and Section 9.

In [ ]:
paired_effects = {}

for ch in CHANNELS:
    paired_effects[ch] = {}
    for sid in SUBJECTS:
        paired_effects[ch][sid] = compare_methods_trial_level(
            summaries[ch][sid]["Video model"],
            summaries[ch][sid]["EOG regression"],
            method_names=("Video model", "EOG regression"),
            use_fisher_z=True,
            n_perm=N_PERM,
            n_boot=N_BOOT,
            seed=SEED,
        )

print(
    f"Trial-level paired comparisons completed for "
    f"{len(CHANNELS) * len(SUBJECTS)} (channel, subject) pairs."
)

### 8.3. Verification of paired-effect sign convention

`compare_methods_trial_level(A, B)` returns:

    Δz(r) = z(r_A) − z(r_B)   (Fisher-z scale)

We assert that in every call A = Video and B = EOG regression.
This guarantees that positive Δz(r) corresponds to the video
model having a higher Fisher-z aggregated correlation than the
EOG regression, and that the back-transformed Δr has the same sign.

Any violation raises an error and stops the notebook. This cell
is diagnostic only and does not produce a manuscript output.

In [ ]:
verification_rows = []

for ch in CHANNELS:
    for sid in SUBJECTS:

        test = paired_effects[ch][sid]

        # ---- label check ----
        label_A = test["methods"]["A"]
        label_B = test["methods"]["B"]
        if label_A != "Video model" or label_B != "EOG regression":
            raise ValueError(
                f"{ch}/S{sid:02d}: unexpected method order "
                f"A={label_A!r}, B={label_B!r}"
            )

        # ---- recomputation from raw trial-level r ----
        rA = np.asarray(
            summaries[ch][sid]["Video model"]["r_trials"], dtype=float
        )
        rB = np.asarray(
            summaries[ch][sid]["EOG regression"]["r_trials"],
            dtype=float,
        )
        mask = np.isfinite(rA) & np.isfinite(rB)

        manual_mean_dz = float(
            np.mean(fisher_z(rA[mask]) - fisher_z(rB[mask]))
        )
        reported_mean_dz = float(test["r"]["summary"]["mean"])

        if not np.isclose(manual_mean_dz, reported_mean_dz, atol=1e-9):
            raise ValueError(
                f"{ch}/S{sid:02d}: reported Δz(r)="
                f"{reported_mean_dz} != manual {manual_mean_dz}"
            )

        # ---- back-transformed Δr sign check ----
        delta_mean_r = float(test["r"]["delta_mean_r"])
        manual_delta_r = float(
            inv_fisher_z(np.mean(fisher_z(rA[mask])))
            - inv_fisher_z(np.mean(fisher_z(rB[mask])))
        )
        if not np.isclose(delta_mean_r, manual_delta_r, atol=1e-9):
            raise ValueError(
                f"{ch}/S{sid:02d}: reported Δr="
                f"{delta_mean_r} != manual {manual_delta_r}"
            )

        sign_ok = (
            (manual_mean_dz > 0 and delta_mean_r > 0) or
            (manual_mean_dz < 0 and delta_mean_r < 0) or
            np.isclose(manual_mean_dz, 0.0, atol=1e-12)
        )
        if not sign_ok:
            raise ValueError(
                f"{ch}/S{sid:02d}: sign inconsistency "
                f"Δz(r)={manual_mean_dz}, Δr={delta_mean_r}"
            )

        verification_rows.append({
            "Subject": sid,
            "Channel": ch,
            "method_A": label_A,
            "method_B": label_B,
            "Delta_z_r": manual_mean_dz,
            "Delta_r":   delta_mean_r,
            "sign_consistent": sign_ok,
        })

df_direction_check = pd.DataFrame(verification_rows)

print(
    f"Direction verification passed for all "
    f"{len(CHANNELS) * len(SUBJECTS)} (channel, subject) pairs."
)
print("Sign convention: positive = Video > EOG.")

### 8.4. Subject-level prediction performance

Descriptive table for each (subject, channel, predictor):

- `r_fisher_z_aggregated` :
    Fisher-z aggregated Pearson r across the 40 held-out
    trials, back-transformed to the r scale.
- `r_CI_low_within_subject` / `r_CI_high_within_subject` :
    Within-subject 95% bootstrap CI over the 40 held-out
    trials. These are **trial-level** CIs and do not
    describe population-level uncertainty.

This table:

- is exported as Supplementary Table S3,
- is used as input for Panel a of the main figure in
    Section 9.

In [ ]:
performance_rows = []

for ch in CHANNELS:
    for sid in SUBJECTS:
        for predictor in ("Video model", "EOG regression"):

            s = summaries[ch][sid][predictor]

            r_lo, r_hi = s["r_pooled_ci_boot"]

            performance_rows.append({
                "Subject":                sid,
                "EEG_target":             ch,
                "Predictor":              predictor,
                "N_trials":               s["trials"],
                "r_fisher_z_aggregated":  s["r_pooled"],
                "r_CI_low_within_subject":         r_lo,
                "r_CI_high_within_subject":        r_hi,
            })

df_subject_performance = (
    pd.DataFrame(performance_rows)
    .sort_values(["EEG_target", "Subject", "Predictor"])
    .reset_index(drop=True)
)

df_subject_performance.to_csv(
    TABLE_DIR / "Table_S3_subject_prediction.csv",
    index=False,
)

print(
    f"Participant-level performance table: "
    f"{len(df_subject_performance)} rows "
    f"(= {len(CHANNELS)} channels × {len(SUBJECTS)} subjects × "
    f"2 predictors)."
)
display(df_subject_performance.round(4))

 ### 8.5. Subject-level spectral reconstruction

Descriptive table for each (subject, channel, predictor) with
two frequency-domain metrics per band (θ, α, β):

- Band-averaged magnitude-squared coherence
- Relative band-power error, defined as
    |RBP_pred − RBP_true|

Both metrics are aggregated across the 40 held-out trials. The
coherence value comes directly from `summarize_trials`; the RBP
error is computed from the per-trial `_raw` arrays stored by
`summarize_trials` and is accompanied by a within-subject
bootstrap CI on the mean.

This table:
- is exported as Supplementary Table S4,
- is used as input for Panel C of the main figure and for
the supplementary spectral figure.


In [ ]:
spectral_rows = []

for ch in CHANNELS:
    for sid in SUBJECTS:
        for predictor in ("Video model", "EOG regression"):

            s = summaries[ch][sid][predictor]

            row = {
                "Subject":    sid,
                "EEG_target": ch,
                "Predictor":  predictor,
            }

            for band in ("theta", "alpha", "beta"):

                bs = s["band_summary"][band]

                # ---- coherence (CI nested under "coh") ----
                row[f"coh_{band}"]         = bs["coh"]["mean_sd_ci"]["mean"]
                row[f"coh_{band}_CI_low_within_subject"]  = bs["coh"]["ci_boot"][0]
                row[f"coh_{band}_CI_high_within_subject"] = bs["coh"]["ci_boot"][1]

                # ---- RBP error computed from per-trial arrays ----
                pred = np.asarray(bs["_raw"]["rbp_pred_trials"], dtype=float)
                true = np.asarray(bs["_raw"]["rbp_true_trials"], dtype=float)
                err  = np.abs(pred - true)
                row[f"rbp_err_{band}"] = (
                    float(np.nanmean(err)) if np.isfinite(err).any()
                    else float(np.nan)
                )
                lo, hi = bootstrap_ci_trial(err, n_boot=N_BOOT, seed=SEED)
                row[f"rbp_err_{band}_CI_low_within_subject"]  = lo
                row[f"rbp_err_{band}_CI_high_within_subject"] = hi

            spectral_rows.append(row)

df_subject_spectral = (
    pd.DataFrame(spectral_rows)
    .sort_values(["EEG_target", "Subject", "Predictor"])
    .reset_index(drop=True)
)

df_subject_spectral.to_csv(
    TABLE_DIR / "Table_S4_subject_spectral.csv",
    index=False,
)

print(
    f"Participant-level spectral table: "
    f"{len(df_subject_spectral)} rows × "
    f"{len(df_subject_spectral.columns)} columns."
)
display(df_subject_spectral.round(4))

### 8.6. Subject-level paired effects

For each (subject, channel), we extract the paired contrasts
computed in Cell 8.2 for every metric:

1) Δz(r)      = z(r_Video) − z(r_EOG)         (Fisher-z)
2) Δcoh_θ,α,β = coh_Video − coh_EOG           (linear)
3) ΔRBP_θ,α,β = |RBP|_Video − |RBP|_EOG       (linear)

The extraction is done via the predefined helper
`comparison_to_dataframes`, which converts the nested output of
`compare_methods_trial_level` into tidy DataFrames:

- "overall"    : Δz(r)
- "coherence"  : per-band coherence
- "rbp_error"  : per-band |RBP_pred − RBP_true|

Each DataFrame already contains the within-subject paired mean,
t-based and bootstrap CIs, sign-flip permutation p, Wilcoxon p,
and Cohen's dz. We only add the (subject, channel) identifiers.

Two long-format tables are produced:

- `df_subject_paired_prediction`  (Δz(r))
- `df_subject_paired_spectral`    (Δcoherence and ΔRBP)

Both are exported as Supplementary tables.

In [ ]:
paired_prediction_rows = []
paired_spectral_rows   = []

for ch in CHANNELS:
    for sid in SUBJECTS:

        # ---- nested structure → tidy DataFrames ----
        dfs = comparison_to_dataframes(paired_effects[ch][sid])

        # ---- (a) Δz(r) ----
        r_row = dfs["overall"].loc["r (Fisher z)"]
        paired_prediction_rows.append({
            "Subject":                sid,
            "EEG_target":             ch,
            "Delta_z_r":              r_row["delta_mean"],
            "CI_low_within_subject":  r_row["ci_boot_low"],
            "CI_high_within_subject": r_row["ci_boot_high"],
            "Cohen_dz_within_subject": r_row["cohen_dz"],
            "Permutation_p_within_subject": r_row["sign_perm_p"],
            "Wilcoxon_p_within_subject":    r_row["wilcoxon_p"],
            "mean_r_Video":           r_row["mean_r_A"],
            "mean_r_EOG":             r_row["mean_r_B"],
        })

        # ---- (b) coherence per band ----
        for band in ("theta", "alpha", "beta"):
            coh_row = dfs["coherence"].loc[band]
            paired_spectral_rows.append({
                "Subject":        sid,
                "EEG_target":     ch,
                "Metric":         "Coherence",
                "Band":           band,
                "Delta":          coh_row["delta_mean"],
                "CI_low":         coh_row["ci_boot_low"],
                "CI_high":        coh_row["ci_boot_high"],
                "Permutation_p":  coh_row["sign_perm_p"],
                "Wilcoxon_p":     coh_row["wilcoxon_p"],
                "Cohen_dz":       coh_row["cohen_dz"],
            })

        # ---- (c) RBP error per band ----
        for band in ("theta", "alpha", "beta"):
            rbp_row = dfs["rbp_error"].loc[band]
            paired_spectral_rows.append({
                "Subject":        sid,
                "EEG_target":     ch,
                "Metric":         "RBP error",
                "Band":           band,
                "Delta":          rbp_row["delta_mean"],
                "CI_low":         rbp_row["ci_boot_low"],
                "CI_high":        rbp_row["ci_boot_high"],
                "Permutation_p":  rbp_row["sign_perm_p"],
                "Wilcoxon_p":     rbp_row["wilcoxon_p"],
                "Cohen_dz":       rbp_row["cohen_dz"],
            })


# ---------------- assemble tidy tables ----------------
df_subject_paired_prediction = (
    pd.DataFrame(paired_prediction_rows)
    .sort_values(["EEG_target", "Subject"])
    .reset_index(drop=True)
)

df_subject_paired_spectral = (
    pd.DataFrame(paired_spectral_rows)
    .sort_values(["Metric", "EEG_target", "Band", "Subject"])
    .reset_index(drop=True)
)

df_subject_paired_prediction.to_csv(
    TABLE_DIR / "Table_S5_subject_paired_prediction.csv",
    index=False,
)
df_subject_paired_spectral.to_csv(
    TABLE_DIR / "Table_S6_subject_paired_spectral.csv",
    index=False,
)

print(
    f"Participant-level paired tables exported: "
    f"prediction = {len(df_subject_paired_prediction)} rows, "
    f"spectral = {len(df_subject_paired_spectral)} rows."
)
display(df_subject_paired_prediction.head(6).round(4))
display(df_subject_paired_spectral.head(12).round(4))

## 9. Paired Video vs EOG: Population-Level Analysis

Purpose:

Aggregate the subject-level effects produced in Section 8 across
the 18 independent participants, and perform population-level
inference on the paired Video − EOG contrasts.

Metrics:

- Fisher-z aggregated Pearson r
- Theta / alpha / beta coherence
- Relative band-power (RBP) error

Statistical hierarchy:

The 18 participants are the independent units. Each participant
contributes one paired effect per (channel, metric, band). All
population statistics are computed on these 18 values:

- mean effect and 95% percentile-bootstrap CI
- two-sided sign-flip permutation p-value
- Wilcoxon signed-rank p-value (secondary)
- Cohen's dz (participant-level effect size)
- Holm-Bonferroni correction within each metric family

Outputs:

    Main:
        Table 1  — Video vs EOG: Fisher-z aggregated correlation
        Table 2  — Video vs EOG: theta-band coherence
        Figure 1 — 3-panel main figure

    Supplementary:
        Table S7 — full population-level spectral contrasts
        Table S8 — channel-specificity contrasts
        Figure S_spectral — alpha/beta coherence and RBP

Sign convention:

All contrasts are A − B = Video − EOG. Positive values indicate
the video model outperforms the EOG regression on the
corresponding metric.

### 9.1. Population-level condition estimates

Descriptive table for each (target channel, predictor):

- `r`          : Fisher-z aggregated Pearson r across 18
                    participants, back-transformed to the r scale.
- `r_CI_low` / `r_CI_high`
                : 95% percentile-bootstrap CI over participants,
                    computed on the Fisher-z scale and
                    back-transformed.

These condition-level estimates feed directly into Panel a of
the main figure and into Table 1.

In [ ]:
def _aggregate_population(values, scale="linear",
                          n_boot=N_BOOT, ci_level=0.95, seed=SEED):
    """
    Aggregate one metric across the 18 independent participants.

    Parameters
    ----------
    values : array-like
        One participant-level value per subject.
    scale : {"linear", "fisher"}
        "fisher" : apply atanh, aggregate on the Fisher-z scale,
                   back-transform for display (for Pearson r).
        "linear" : aggregate on the raw scale (for RMSE, coherence).
    """
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    n = v.size
    if n == 0:
        return dict(n=0, mean=np.nan, ci_lo=np.nan, ci_hi=np.nan)

    v_t = fisher_z(v) if scale == "fisher" else v
    mean_t = float(np.mean(v_t))

    if n > 1:
        boot = stats.bootstrap(
            (v_t,), np.mean,
            confidence_level=ci_level,
            n_resamples=n_boot,
            method="percentile",
            random_state=seed,
        )
        ci_lo_t = float(boot.confidence_interval.low)
        ci_hi_t = float(boot.confidence_interval.high)
    else:
        ci_lo_t = ci_hi_t = mean_t

    if scale == "fisher":
        mean_disp  = float(inv_fisher_z(mean_t))
        ci_lo_disp = float(inv_fisher_z(ci_lo_t))
        ci_hi_disp = float(inv_fisher_z(ci_hi_t))
    else:
        mean_disp, ci_lo_disp, ci_hi_disp = mean_t, ci_lo_t, ci_hi_t

    return dict(n=n, mean=mean_disp, ci_lo=ci_lo_disp, ci_hi=ci_hi_disp)


# ============================================================
# condition-level estimates
# ============================================================
condition_rows = []

for ch in CHANNELS:
    for predictor in ("Video model", "EOG regression"):

        r_values = np.array([
            summaries[ch][sid][predictor]["r_pooled"]
            for sid in SUBJECTS
        ], dtype=float)

        r_agg = _aggregate_population(r_values, scale="fisher")

        condition_rows.append({
            "Target":      ch,
            "Predictor":   predictor,
            "N subjects":  r_agg["n"],
            "r":           r_agg["mean"],
            "r_CI_low":    r_agg["ci_lo"],
            "r_CI_high":   r_agg["ci_hi"],
        })

df_population_conditions = pd.DataFrame(condition_rows)

display(df_population_conditions.round(4))

### 9.2. Population-level paired effect: Δz(r)

For each EEG target channel:
    Δz(r) = z(r_Video) − z(r_EOG)    (Fisher-z scale)

Population statistics are computed on the 18 participant-level
paired effects:

- mean and 95% percentile-bootstrap CI over participants
- two-sided sign-flip permutation p-value
- Wilcoxon signed-rank p-value
- Cohen's dz (participant-level effect size)
- Holm-Bonferroni adjusted p-value across the three channels

In [ ]:
effect_rows = []

for ch in CHANNELS:

    # ---- participant-level Δz(r) from Section 8 ----
    r_effects = np.array([
        paired_effects[ch][sid]["r"]["summary"]["mean"]
        for sid in SUBJECTS
    ], dtype=float)

    v = r_effects[np.isfinite(r_effects)]
    if v.size == 0:
        continue

    boot = stats.bootstrap(
        (v,), np.mean,
        confidence_level=0.95,
        n_resamples=N_BOOT,
        method="percentile",
        random_state=SEED,
    )
    p_perm = sign_flip_pvalue(v, n_perm=N_PERM, seed=SEED)

    try:
        p_wilcox = float(stats.wilcoxon(
            v, alternative="two-sided", zero_method="wilcox"
        ).pvalue)
    except ValueError:
        p_wilcox = np.nan

    effect_rows.append({
        "Target":        ch,
        "N subjects":    v.size,
        "Delta_z_r":     float(np.mean(v)),
        "CI_low":        float(boot.confidence_interval.low),
        "CI_high":       float(boot.confidence_interval.high),
        "Cohen_dz":      population_cohens_dz(v),
        "Permutation_p": p_perm,
        "Wilcoxon_p":    p_wilcox,
    })

df_population_delta_z_r = pd.DataFrame(effect_rows)

df_population_delta_z_r["Holm_p"] = multipletests(
    df_population_delta_z_r["Permutation_p"], method="holm"
)[1]

df_population_delta_z_r = df_population_delta_z_r[[
    "Target", "N subjects", "Delta_z_r",
    "CI_low", "CI_high", "Cohen_dz",
    "Permutation_p", "Wilcoxon_p", "Holm_p",
]]

display(df_population_delta_z_r.round(5))

### 9.3. Population-level spectral effects

For each (target channel, band), we aggregate the participant-
level paired effects computed in Section C for two spectral
metrics:

- Δcoherence   = coh_Video − coh_EOG          (linear scale)
- ΔRBP_error   = |ΔRBP|_Video − |ΔRBP|_EOG    (linear scale)

Band = theta / alpha / beta. Theta is the primary frequency-
domain band, because the dominant spectral component of the Fz
target lies in the theta range. Alpha and beta are reported as
secondary metrics in the Supplementary Material.

Holm correction is applied within each (metric, band) family
(3 channels per family). This yields separate families for
theta-coherence, alpha-coherence, beta-coherence,
theta-RBP, alpha-RBP, and beta-RBP.

In [ ]:
spectral_effect_rows = []

for ch in CHANNELS:
    for band in ("theta", "alpha", "beta"):

        # ---- participant-level paired effects from Section C ----
        coh_effects = np.array([
            paired_effects[ch][sid]["coherence"]["per_band"][band]["summary"]["mean"]
            for sid in SUBJECTS
        ], dtype=float)

        rbp_effects = np.array([
            paired_effects[ch][sid]["rbp_error"]["per_band"][band]["summary"]["mean"]
            for sid in SUBJECTS
        ], dtype=float)

        for metric_name, effects in (
            ("Coherence", coh_effects),
            ("RBP error", rbp_effects),
        ):
            v = effects[np.isfinite(effects)]
            if v.size == 0:
                continue

            boot = stats.bootstrap(
                (v,), np.mean,
                confidence_level=0.95,
                n_resamples=N_BOOT,
                method="percentile",
                random_state=SEED,
            )
            p_perm = sign_flip_pvalue(v, n_perm=N_PERM, seed=SEED)

            try:
                p_wilcox = float(stats.wilcoxon(
                    v, alternative="two-sided", zero_method="wilcox"
                ).pvalue)
            except ValueError:
                p_wilcox = np.nan

            spectral_effect_rows.append({
                "Target":        ch,
                "Band":          band,
                "Metric":        metric_name,
                "N subjects":    v.size,
                "Effect":        float(np.mean(v)),
                "CI_low":        float(boot.confidence_interval.low),
                "CI_high":       float(boot.confidence_interval.high),
                "Cohen_dz":      population_cohens_dz(v),
                "Permutation_p": p_perm,
                "Wilcoxon_p":    p_wilcox,
            })

df_population_spectral = pd.DataFrame(spectral_effect_rows)

# ---- Holm correction within each (metric, band) family ----
df_population_spectral["Holm_p"] = np.nan
for (metric, band) in df_population_spectral[["Metric", "Band"]].drop_duplicates().itertuples(index=False):
    mask = (
        df_population_spectral["Metric"].eq(metric) &
        df_population_spectral["Band"].eq(band)
    )
    p = df_population_spectral.loc[mask, "Permutation_p"].to_numpy(float)
    valid = np.isfinite(p)
    adj = np.full(len(p), np.nan)
    if valid.any():
        adj[valid] = multipletests(p[valid], method="holm")[1]
    df_population_spectral.loc[mask, "Holm_p"] = adj

df_population_spectral = (
    df_population_spectral[[
        "Target", "Band", "Metric", "N subjects", "Effect",
        "CI_low", "CI_high", "Cohen_dz",
        "Permutation_p", "Wilcoxon_p", "Holm_p",
    ]]
    .sort_values(["Metric", "Band", "Target"])
    .reset_index(drop=True)
)

display(df_population_spectral.round(5))

### 9.4. Channel-specificity contrasts (three families)
All contrasts are computed in Fisher-z space and tested
pairwise across the 18 participants.

   1) Video absolute : z(r_Video,Fz) − z(r_Video,ch2)
   2) EOG absolute   : z(r_EOG,Fz)   − z(r_EOG,ch2)
   3) Differential   : Δz(r)_Fz      − Δz(r)_ch2

where ch2 ∈ {Pz, Oz}. Pz−Oz is reported in the Supplementary.

In [ ]:
def _channel_contrast(values_a, values_b):
    d = np.asarray(values_a, dtype=float) - np.asarray(values_b, dtype=float)
    d = d[np.isfinite(d)]
    boot = stats.bootstrap(
        (d,), np.mean, confidence_level=0.95,
        n_resamples=N_BOOT, method="percentile", random_state=SEED,
    )
    p_perm = sign_flip_pvalue(d, n_perm=N_PERM, seed=SEED)
    try:
        p_wilcox = float(stats.wilcoxon(
            d, alternative="two-sided", zero_method="wilcox",
        ).pvalue)
    except ValueError:
        p_wilcox = np.nan
    return {
        "N subjects":    d.size,
        "Effect":        float(np.mean(d)),
        "CI_low":        float(boot.confidence_interval.low),
        "CI_high":       float(boot.confidence_interval.high),
        "Cohen_dz":      population_cohens_dz(d),
        "Permutation_p": p_perm,
        "Wilcoxon_p":    p_wilcox,
    }

# ---- per-subject Fisher-z values ----
z_video = {ch: np.array([
    fisher_z(summaries[ch][sid]["Video model"]["r_pooled"]) for sid in SUBJECTS
], dtype=float) for ch in CHANNELS}

z_eog = {ch: np.array([
    fisher_z(summaries[ch][sid]["EOG regression"]["r_pooled"])
    for sid in SUBJECTS
], dtype=float) for ch in CHANNELS}

delta_z = {ch: np.array([
    paired_effects[ch][sid]["r"]["summary"]["mean"]
    for sid in SUBJECTS
], dtype=float) for ch in CHANNELS}

# ---- main contrasts (Fz vs posterior only) ----
contrast_rows = []
for ch2 in ("Pz", "Oz"):
    for family, va, vb in (
        ("Video absolute",   z_video["Fz"], z_video[ch2]),
        ("EOG absolute",     z_eog["Fz"],   z_eog[ch2]),
        ("Differential",     delta_z["Fz"], delta_z[ch2]),
    ):
        row = _channel_contrast(va, vb)
        row["Family"]   = family
        row["Contrast"] = f"Fz − {ch2}"
        contrast_rows.append(row)

df_channel_contrasts = pd.DataFrame(contrast_rows)

# ---- Holm correction within each family ----
df_channel_contrasts["Holm_p"] = np.nan
for family in df_channel_contrasts["Family"].unique():
    mask = df_channel_contrasts["Family"].eq(family)
    p = df_channel_contrasts.loc[mask, "Permutation_p"].to_numpy(float)
    valid = np.isfinite(p)
    adj = np.full(len(p), np.nan)
    if valid.any():
        adj[valid] = multipletests(p[valid], method="holm")[1]
    df_channel_contrasts.loc[mask, "Holm_p"] = adj

# ---- supplementary: Pz − Oz for completeness ----
contrast_supp_rows = []
for family, va, vb in (
    ("Video absolute", z_video["Pz"], z_video["Oz"]),
    ("EOG absolute",   z_eog["Pz"],   z_eog["Oz"]),
    ("Differential",   delta_z["Pz"], delta_z["Oz"]),
):
    row = _channel_contrast(va, vb)
    row["Family"]   = family
    row["Contrast"] = "Pz − Oz"
    contrast_supp_rows.append(row)

df_channel_contrasts_supp = pd.DataFrame(contrast_supp_rows)

df_channel_contrasts_supp.to_csv(
    TABLE_DIR / "Table_S8b_channel_specificity_supp.csv", index=False
)
display(df_channel_contrasts.round(5))
display(df_channel_contrasts_supp.round(5))

### 9.5. Main Table 1 — Video vs EOG: Fisher-z aggregated correlation

Combined primary-results table across the three EEG target
channels. Condition-level estimates are reported as Fisher-z
aggregated Pearson r (back-transformed). The paired contrast is
reported as Δz(r) on the Fisher-z scale.

Amplitude-domain error (RMSE) is not included in this table:
for standardized target signals, RMSE is mathematically coupled
to Pearson r (RMSE ≈ √(1 − r²)) and does not provide independent
evidence. See Supplementary Table S9 for a diagnostic report.

In [ ]:
main_table1_rows = []

for ch in CHANNELS:

    v_row = df_population_conditions[
        (df_population_conditions["Target"] == ch) &
        (df_population_conditions["Predictor"] == "Video model")
    ].iloc[0]

    e_row = df_population_conditions[
        (df_population_conditions["Target"] == ch) &
        (df_population_conditions["Predictor"] == "EOG regression")
    ].iloc[0]

    eff = df_population_delta_z_r[
        df_population_delta_z_r["Target"] == ch
    ].iloc[0]

    main_table1_rows.append({
        "Target":      ch,
        "r_Video":     v_row["r"],
        "r_Video_lo":  v_row["r_CI_low"],
        "r_Video_hi":  v_row["r_CI_high"],
        "r_EOG":       e_row["r"],
        "r_EOG_lo":    e_row["r_CI_low"],
        "r_EOG_hi":    e_row["r_CI_high"],
        "Delta_z_r":   eff["Delta_z_r"],
        "Delta_lo":    eff["CI_low"],
        "Delta_hi":    eff["CI_high"],
        "Cohen_dz":    eff["Cohen_dz"],
        "P_Holm":      eff["Holm_p"],
    })

df_main_table1 = pd.DataFrame(main_table1_rows)

df_main_table1.to_csv(
    TABLE_DIR / "Table_1_main_video_vs_eog.csv",
    index=False,
)

display(df_main_table1.round(4))

### 9.6. Main Table 2 — Video vs EOG: theta-band coherence

Theta-band coherence is reported as the primary
frequency-domain metric because the dominant spectral component
of the Fz target lies in the theta range. Condition-level
values and paired contrasts are computed on the same 18
participants and channels, using the same statistical framework
as in Table 1.


In [ ]:
theta_rows = []

for ch in CHANNELS:

    # ---- condition-level theta coherence ----
    v_vals = np.array([
        summaries[ch][sid]["Video model"]["band_summary"]["theta"]["coh"]["mean_sd_ci"]["mean"]
        for sid in SUBJECTS
    ], dtype=float)
    e_vals = np.array([
        summaries[ch][sid]["EOG regression"]["band_summary"]["theta"]["coh"]["mean_sd_ci"]["mean"]
        for sid in SUBJECTS
    ], dtype=float)

    v_agg = _aggregate_population(v_vals, scale="linear")
    e_agg = _aggregate_population(e_vals, scale="linear")

    # ---- paired contrast from Section 9.3 ----
    eff = df_population_spectral[
        (df_population_spectral["Target"] == ch) &
        (df_population_spectral["Band"] == "theta") &
        (df_population_spectral["Metric"] == "Coherence")
    ].iloc[0]

    theta_rows.append({
        "Target":          ch,
        "coh_Video":       v_agg["mean"],
        "coh_Video_lo":    v_agg["ci_lo"],
        "coh_Video_hi":    v_agg["ci_hi"],
        "coh_EOG":         e_agg["mean"],
        "coh_EOG_lo":      e_agg["ci_lo"],
        "coh_EOG_hi":      e_agg["ci_hi"],
        "Delta_theta":     eff["Effect"],
        "Delta_theta_lo":  eff["CI_low"],
        "Delta_theta_hi":  eff["CI_high"],
        "Delta_theta_dz":  eff["Cohen_dz"],
        "Delta_theta_P":   eff["Holm_p"],
    })

df_main_table2 = pd.DataFrame(theta_rows)

df_main_table2.to_csv(
    TABLE_DIR / "Table_2_main_theta_coherence.csv",
    index=False,
)

display(df_main_table2.round(4))

### 9.7. Supplementary tables export

Two supplementary tables are exported:

- Table S7 : full population-level spectral contrasts
                (coherence and RBP for theta, alpha, beta)
- Table S8 : channel-specificity contrasts of Δz(r)

In [ ]:
# ---- Table S7 : full spectral contrasts ----
df_population_spectral.to_csv(
    TABLE_DIR / "Table_S7_population_spectral.csv",
    index=False,
)

# ---- Table S8 : channel specificity ----
df_channel_contrasts.to_csv(
    TABLE_DIR / "Table_S8_channel_specificity.csv",
    index=False,
)

print("Supplementary tables exported:")
for f in ("Table_S7_population_spectral.csv",
          "Table_S8_channel_specificity.csv"):
    print(f"  {TABLE_DIR / f}")

### 9.8. Main Figure — Video vs EOG across EEG target channels

Three panels:

 - Panel a — Fisher-z aggregated Pearson r
    Video vs EOG regression, per EEG target channel.

 - Panel b — Incremental correlation over EOG:
    Δz(r) = z(r_Video) − z(r_EOG), per channel.
    Holm-adjusted P-values are annotated on the figure;
    effect sizes (Cohen's dz) are reported in Table 1.

 - Panel c — Incremental coherence over EOG:
    ΔCoh = Coh_Video − Coh_EOG, per channel x band.
    Bands: theta (4-8 Hz), alpha (8-13 Hz), beta
    (13-25 Hz). Fisher-z is NOT applied to coherence.
 - Panel d — Incremental RBP error over EOG (ΔRBP, three bands)

All panels display 18 subject-level observations per condition,
with the population mean and 95% bootstrap CI overlaid.

In [ ]:
fig, axes = plot_main_figure_video_vs_eog(
    df_performance           = df_subject_performance,
    df_spectral              = df_subject_spectral,
    df_population_conditions = df_population_conditions,
    df_population_delta_z_r  = df_population_delta_z_r,
    df_population_spectral   = df_population_spectral,
    subjects                 = SUBJECTS,
    channels                 = CHANNELS,
    bands                    = ("theta", "alpha", "beta"),
    save_fig                 = True,
    file_name                = str(FIG_DIR / "fig_main_video_vs_eog.pdf"),
)
plt.show()
plt.close(fig)

### 9.9. Supplementary spectral figure

Extended spectral panel covering the four metrics that are not
part of the main figure:

- alpha coherence
- beta coherence
- theta RBP error
- alpha RBP error
- beta RBP error

Each panel displays the same paired structure as Panel a-c of
the main figure, using participant-level values from Section C.

In [ ]:
fig, axes = plot_supplementary_spectral_heatmaps(
    df_spectral=df_subject_spectral,
    subjects=SUBJECTS,
    channels=("Fz", "Pz", "Oz"),
    bands=("theta", "alpha", "beta"),
    save_fig=True,
    file_name=str(
        FIG_DIR / "fig_S_spectral_heatmaps.pdf"
    ),
    annotate=True,
    fmt=".2f",
)

plt.show()
plt.close(fig)